## Token Embedding
Now that we understand tokenization, we have solved one part of the problem of representing language to a language model. In this sense, language is a sequence of tokens. And if we train a good-enough model on a large-enough set of tokens, it starts to capture the complex patterns that appear in its training dataset:
 - If the training data contains a lot of English text, that pattern reveals itself as a model capable of representing and generating the English language
 - If the training data contains factual information (Wikipedia, for example), the model would have the ability to generate some factual information (see the following note).

### A Language Model Holds Embeddings for the Vocabulary of Its Tokenizer

- After a tokenizer is initialized and trained, it is then used in the training process of its associated language model.

- This is why a pretrained language model is linked with its tokenizer and can’t use a different tokenizer without training.
- The language model holds an embedding vector for each token in the tokenizer’s vocabulary
- Before the beginning of the training process, these vectors are randomly initialized like the rest of the model’s weights, but the training process assigns them the values that enable the useful behavior they’re trained to perform.



### Creating Contextualized Word Embeddings with Language Models

In [1]:
from transformers import AutoModel, AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")


# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/a

In [2]:
output.shape

torch.Size([1, 4, 384])

- If we skip the first dimension, we can read this as 4 tokens, each token is embedded in a vector of 384 values.
- The first dimension is the batch dimension used in cases (like training) when we want to send multiple input sentences to the model at the same time (they’re processed at the same time, which speeds up the process).

In [3]:
for token  in tokens['input_ids'][0]:
    print(tokenizer.decode([token]))

[CLS]
Hello
 world
[SEP]


- This particular tokenizer and model operate by adding the [CLS] and [SEP] tokens to the beginning and end of a string.
- Our language model has now processed the text input. The result of its output is the following:

In [4]:
output

tensor([[[-3.4824,  0.0856, -0.1815,  ..., -0.0614, -0.3911,  0.3030],
         [ 0.1912,  0.3196, -0.2316,  ...,  0.3735,  0.2477,  0.8042],
         [ 0.2076,  0.5024, -0.0485,  ...,  1.2197, -0.2281,  0.8540],
         [-3.4277,  0.0643, -0.1425,  ...,  0.0657, -0.4365,  0.3835]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

## Word Embedding Beyond LLMs

In [5]:
import gensim.downloader as api
api.info

model.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

<function gensim.downloader.info(name=None, show_only_latest=True, name_only=False)>

d:\2026-courses\LLMs-Handson\venv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub\models--microsoft--deberta-v3-xsmall. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [6]:
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [7]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

## Recommending songs by embeddings

In [19]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [16]:
songs_df.head()

,title,artist
id,,
0,Gucci Time (w\/ Swizz Beatz),Gucci Mane
1,Aston Martin Music (w\/ Drake & Chrisette Mich...,Rick Ross
2,Get Back Up (w\/ Chris Brown),T.I.
3,Hot Toddy (w\/ Jay-Z & Ester Dean),Usher
4,Whip My Hair,Willow


In [20]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [21]:
from gensim.models import Word2Vec
# Train our Word2Vec
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

- playlists: The training corpus; each element should be a list (e.g., a playlist is a list of song IDs or tokens). The model learns that items appearing in similar playlists should have similar vectors.
- vector_size=32: Each item is mapped to a 32‑dimensional embedding vector; larger size can capture more nuance but is slower and may overfit.
- window=20: The context window size is 20, meaning for each target item, the model looks up to 20 items before and after it in a playlist to define “context.” With playlists, this says “items within up to 20 positions in the same playlist are related.”
- negative=50: Uses negative sampling with 50 negative examples per positive pair. This greatly speeds up training by comparing each true (target, context) pair with 50 randomly sampled “non‑context” items, pushing their vectors apart.
- min_count=1: Keep all items that appear at least once; no item is discarded for low frequency. This is common when the “vocabulary” is finite and every ID matters (e.g., songs).
- workers=4: Use 4 parallel worker threads to train the model on multiple CPU cores, reducing training time.

In [22]:
song_id = 2172

#Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('2849', 0.9983966946601868),
 ('5586', 0.9969850182533264),
 ('11596', 0.9964770674705505),
 ('3167', 0.9962591528892517),
 ('2976', 0.9961287379264832),
 ('5634', 0.9959601759910583),
 ('3116', 0.9957323670387268),
 ('1922', 0.9956367015838623),
 ('2014', 0.9950476288795471),
 ('3148', 0.9948804378509521)]

In [23]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [32]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)



,title,artist
id,,
2849,Run To The Hills,Iron Maiden
5586,The Last In Line,Dio
11596,Hallowed Be Thy Name,Iron Maiden
3167,Unchained,Van Halen
2976,I Don't Know,Ozzy Osbourne


In [33]:
import numpy as np
import pandas as pd  # if not already imported

def print_recommendations(song_id, topn=5):
    # List of (similar_id, similarity_score) tuples
    sims = model.wv.most_similar(positive=str(song_id), topn=topn)

    # Convert to a NumPy array: shape (topn, 2)
    sims_array = np.array(sims)

    # First column = IDs, second column = scores
    similar_ids = sims_array[:, 0]          # ['songA', 'songB', ...]
    similarity_scores = sims_array[:, 1]    # [0.95, 0.93, ...]

    # Select song rows; ensure indices match how IDs are stored
    recs = songs_df.iloc[similar_ids.astype(int)]

    # Attach scores as a new column
    recs = recs.copy()
    recs["similarity"] = similarity_scores

    return recs

In [34]:
print_recommendations(2172)

,title,artist,similarity
id,,,
2849,Run To The Hills,Iron Maiden,0.9983966946601868
5586,The Last In Line,Dio,0.9969850182533264
11596,Hallowed Be Thy Name,Iron Maiden,0.9964770674705505
3167,Unchained,Van Halen,0.9962591528892517
2976,I Don't Know,Ozzy Osbourne,0.9961287379264832
